In [2]:
AA_SEQS_MERGE_E="../resources/merged_data/AA_sequences/E_DENV{}_aa_E.fasta"
TITER_SUB_MODEL_MERGE = "../resources/merged_data/titers/titer_sub_DENV{}.json"

## Cargar Mapapa Antigenico

In [5]:
mapa_ag = pd.read_csv(
    "../resources/experimental_data/2d-antigenic-map.txt",
    sep=r"\s+",
    header=None,
    names=["type", "strainID", "coord1", "coord2"]
)

# Extraer serotipo: acepta DENV1 y DEN1
mapa_ag["sero_num"] = mapa_ag["strainID"].str.extract(
    r"(?:DENV|DEN)(\d)",
    expand=False
)

mapa_ag["sero"] = "DENV" + mapa_ag["sero_num"]
antigenos = mapa_ag[mapa_ag["type"] == "AG"].copy()
sueros = mapa_ag[mapa_ag["type"] == "SR"].copy()

print(f"Antígenos: {len(antigenos)}")
print(f"Antisueros: {len(sueros)}")

Antígenos: 412
Antisueros: 36


## Titulos predichos para las secuencias colombianas (modelo `sub`)

In [3]:
# ===========================================================================
# Titulos predichos para las secuencias colombianas (modelo sub)
#
# augur NO escribe esto: el CLI nunca llama predict_titer para pares no
# medidos. Pero todos los ingredientes estan en el JSON del modelo sub, asi
# que lo reconstruimos. La formula es la de SubstitutionModel.predict_titer:
#
#     pred = avidez[virus] + potencia[ref][suero] + SUM(efecto de cada
#            sustitucion de aminoacido entre la cepa de referencia del suero
#            y el virus de prueba)
#
# Orden de trabajo, y el orden importa:
#   1. Reconstruir y VALIDAR contra los pares medidos. Si no los reproduce,
#      la reconstruccion esta mal y no se sigue.
#   2. Anclar la escala a titulos crudos usando esos mismos pares medidos,
#      en vez de asumir la formula de normalizacion de augur.
#   3. Recien entonces predecir para las colombianas.
#
# Supuesto que queda a la vista: para un virus no medido la avidez no existe
# en el modelo y vale 0, igual que hace augur. El bloque 2 del analisis
# exploratorio cuantifica cuanto error arrastra eso.
# ===========================================================================

import json
import numpy as np
import pandas as pd
from Bio import SeqIO
# OJO: este notebook importa matplotlib.path.Path en la celda de imports,
# asi que 'Path' NO es pathlib aqui. Alias explicito para rutas de archivo.
from pathlib import Path as FsPath

GENE = "E"


def load_aa(path):
    return {r.id: str(r.seq) for r in SeqIO.parse(path, "fasta")}


def mutations(seq_ref, seq_test):
    """Mismo criterio que TiterModel.get_mutations: posicion 1-based, 'A123B'."""
    return [f"{GENE}:{a}{i + 1}{b}"
            for i, (a, b) in enumerate(zip(seq_ref, seq_test)) if a != b]


def predict(virus, ref, serum, aa, potency, avidity, substitution):
    """Devuelve (prediccion, n_subs_con_efecto) o (None, 0) si falta secuencia."""
    if virus not in aa or ref not in aa:
        return None, 0
    pot = potency.get(ref, {}).get(serum)
    if pot is None:
        return None, 0
    avi = avidity.get(virus, 0.0)          # 0.0 para todo virus no medido
    used = [substitution[m] for m in mutations(aa[ref], aa[virus])
            if m in substitution]
    return avi + pot + float(np.sum(used)), len(used)


def sera_of(potency):
    """potency es ref -> {serum_id: valor, 'mean_potency': valor}."""
    return [(ref, s) for ref, d in potency.items()
            for s in d if s != "mean_potency"]


pred_frames, anchor_frames, recon_report = [], [], []

for sero in range(1, 5):
    try:
        with open(TITER_SUB_MODEL_MERGE.format(sero)) as fh:
            model = json.load(fh)
        aa = load_aa(AA_SEQS_MERGE_E.format(sero))
    except FileNotFoundError as e:
        print(f"DENV{sero}: falta {e.filename}, lo salto")
        continue

    potency = model.get("potency", {}) or {}
    avidity = model.get("avidity", {}) or {}
    substitution = model.get("substitution", {}) or {}
    measured = model.get("titers", {}) or {}

    if not substitution or not potency:
        print(f"DENV{sero}: modelo sin sustituciones o sin potencias, lo salto")
        continue

    # -------------------------------------- 1. error de ajuste sobre lo medido
    # OJO: measured['titers'] guarda las MEDICIONES (valor normalizado +
    # mediana cruda), no predicciones del modelo. Asi que esto no es un
    # round-trip exacto: es el error de ajuste del modelo, y se juzga contra
    # el modelo NULO (solo potencia + avidez, sin sustituciones).
    # Si el modelo no le gana al nulo, la formula reconstruida esta mal
    # o el modelo no aprendio nada.
    obs, rec, nul = [], [], []
    for ref, tests in measured.items():
        for test, sera in tests.items():
            for serum, val in sera.items():
                target = val[0] if isinstance(val, (list, tuple)) else val
                p, _ = predict(test, ref, serum, aa, potency, avidity, substitution)
                pot0 = potency.get(ref, {}).get(serum)
                if p is not None and target is not None and pot0 is not None:
                    obs.append(float(target))
                    rec.append(p)
                    nul.append(avidity.get(test, 0.0) + pot0)
    obs, rec, nul = np.array(obs), np.array(rec), np.array(nul)
    if obs.size == 0:
        print(f"DENV{sero}: no pude recomponer ningun par medido, lo salto")
        continue
    err = rec - obs
    rmse = float(np.sqrt((err ** 2).mean()))
    rmse_null = float(np.sqrt(((nul - obs) ** 2).mean()))
    slope = float(np.polyfit(obs, rec, 1)[0]) if np.ptp(obs) > 0 else np.nan
    recon_report.append(dict(serotype=sero, n_pares=obs.size,
                             rmse=rmse, rmse_nulo=rmse_null,
                             ganancia=rmse_null - rmse,
                             sesgo=float(err.mean()), pendiente=slope))

    # ------------------------------------- 2. anclar la escala a titulos crudos
    # Para cada par medido el JSON trae [valor_normalizado, mediana_cruda].
    # Si el valor normalizado es una caida respecto al homologo, entonces
    #     log2(crudo) = ancla_suero - normalizado
    # El ancla se estima por suero y su SD residual dice si el supuesto se sostiene.
    anchors = {}
    for ref, tests in measured.items():
        for test, sera in tests.items():
            for serum, val in sera.items():
                if not isinstance(val, (list, tuple)) or len(val) < 2:
                    continue
                norm, raw = val[0], val[1]
                if raw is None or raw <= 0:
                    continue
                anchors.setdefault((ref, serum), []).append(np.log2(raw) + float(norm))
    anchor_mu = {k: float(np.mean(v)) for k, v in anchors.items()}
    anchor_sd = {k: (float(np.std(v, ddof=1)) if len(v) > 1 else np.nan)
                 for k, v in anchors.items()}
    sd_vals = np.array([v for v in anchor_sd.values() if not np.isnan(v)])
    anchor_frames.append(dict(
        serotype=sero, n_sueros=len(anchor_mu),
        sd_mediana=float(np.median(sd_vals)) if sd_vals.size else np.nan,
        sd_max=float(sd_vals.max()) if sd_vals.size else np.nan))

    # ------------------------------------------- 3. predecir para colombianas
    col = [s for s in aa if "colombia" in s.lower()]
    rows = []
    for ref, serum in sera_of(potency):
        for virus in col:
            p, n_used = predict(virus, ref, serum, aa, potency, avidity, substitution)
            if p is None:
                continue
            key = (ref, serum)
            titer = (2 ** (anchor_mu[key] - p)) if key in anchor_mu else np.nan
            rows.append(dict(serotype=sero, strain=virus, ref_virus=ref,
                             serum_id=serum, pred_norm_log2=p,
                             pred_titer=titer, n_subs=n_used,
                             avidity_assumed=avidity.get(virus, 0.0)))
    if rows:
        pred_frames.append(pd.DataFrame(rows))

# ================================================================= reportes
print("=" * 78)
print("1. VALIDACION: la reconstruccion reproduce los pares MEDIDOS?")
print("=" * 78)
rep = pd.DataFrame(recon_report)
print(rep.to_string(index=False) if len(rep) else "  (sin datos)")
print("\n'rmse' es el error de ajuste del modelo sobre pares MEDIDOS, en log2.")
print("Se juzga contra 'rmse_nulo' (solo potencia + avidez, sin sustituciones):")
print("  ganancia > 0  -> las sustituciones aportan; la formula es correcta.")
print("  ganancia <= 0 -> el modelo no aprendio nada util, o la formula esta mal.")
print("'sesgo' debe rondar 0 y 'pendiente' debe rondar 1; si no, hay error de")
print("signo o de escala en la reconstruccion.")
print("Referencia: la reproducibilidad del ensayo ronda 1 log2, asi que un")
print("rmse por debajo de ~1 es un ajuste razonable, no un problema.")

print("\n" + "=" * 78)
print("2. ANCLAJE DE ESCALA: se sostiene log2(crudo) = ancla - normalizado?")
print("=" * 78)
anc = pd.DataFrame(anchor_frames)
print(anc.to_string(index=False) if len(anc) else "  (sin datos)")
print("\n'sd_mediana' es la dispersion del ancla dentro de un mismo suero.")
print("Cerca de 0 = la relacion se sostiene y 'pred_titer' es interpretable.")
print("Grande = el supuesto de normalizacion no aplica; usa solo pred_norm_log2.")

if pred_frames:
    pred = pd.concat(pred_frames, ignore_index=True)
    out = "../output/predicted_titers_colombia_sub.csv"
    FsPath(out).parent.mkdir(parents=True, exist_ok=True)
    pred.to_csv(out, index=False)

    print("\n" + "=" * 78)
    print("3. PREDICCIONES PARA SECUENCIAS COLOMBIANAS")
    print("=" * 78)
    print(f"{len(pred)} predicciones -> {out}\n")
    print(f"{'':7s} {'secuencias':>10s} {'sueros':>7s} {'E unicas':>9s} "
          f"{'valores unicos':>15s} {'sin subs':>9s}")
    for sero, g in pred.groupby("serotype"):
        n_seq = g["strain"].nunique()
        n_ser = g["serum_id"].nunique()
        # techo: cuantas proteinas E distintas hay entre esas colombianas
        aa = load_aa(AA_SEQS_MERGE_E.format(sero))
        n_prot = len({aa[s] for s in g["strain"].unique() if s in aa})
        # real: valores distintos que produce el modelo, por suero
        uniq = (g.groupby("serum_id")["pred_norm_log2"]
                 .apply(lambda v: v.round(6).nunique()).median())
        no_sub = int((g["n_subs"] == 0).sum())
        print(f"DENV{sero}  {n_seq:10d} {n_ser:7d} {n_prot:9d} "
              f"{uniq:15.0f} {no_sub:9d}")

    print("\n'E unicas' es el techo: secuencias con la misma proteina E dan")
    print("forzosamente el mismo titulo. 'valores unicos' es lo que de verdad")
    print("produce el modelo (mediana entre sueros) y siempre es <= ese techo,")
    print("porque las diferencias en posiciones con efecto 0 tambien colapsan.")
    print("'sin subs' son predicciones donde NINGUNA sustitucion tenia efecto")
    print("ajustado: ahi la prediccion es solo potencia del suero, es decir el")
    print("modelo nulo. Si ese numero es alto, no hay senal para esas cepas.")

    print("\nIncertidumbre minima por prediccion (SD de avidez, en log2):")
    for sero in sorted(pred["serotype"].unique()):
        with open(TITER_SUB_MODEL_MERGE.format(sero)) as fh:
            avi = np.array(list((json.load(fh).get("avidity", {}) or {}).values()),
                           dtype=float)
        if avi.size > 1:
            sd = avi.std(ddof=1)
            print(f"  DENV{sero}: +/- {sd:.2f} log2  (~{2 ** sd:.1f}x en titulo)")
else:
    pred = pd.DataFrame()
    print("\nNo se genero ninguna prediccion.")


1. VALIDACION: la reconstruccion reproduce los pares MEDIDOS?
 serotype  n_pares     rmse  rmse_nulo  ganancia     sesgo  pendiente
        1      524 0.477539   0.533627  0.056088  0.001954   0.798161
        2      328 0.635468   0.796946  0.161478 -0.003421   0.573010
        3      405 0.530235   0.855240  0.325005 -0.001037   0.788429
        4      305 0.735779   1.011153  0.275374  0.004324   0.473191

'rmse' es el error de ajuste del modelo sobre pares MEDIDOS, en log2.
Se juzga contra 'rmse_nulo' (solo potencia + avidez, sin sustituciones):
  ganancia > 0  -> las sustituciones aportan; la formula es correcta.
  ganancia <= 0 -> el modelo no aprendio nada util, o la formula esta mal.
'sesgo' debe rondar 0 y 'pendiente' debe rondar 1; si no, hay error de
signo o de escala en la reconstruccion.
Referencia: la reproducibilidad del ensayo ronda 1 log2, asi que un
rmse por debajo de ~1 es un ajuste razonable, no un problema.

2. ANCLAJE DE ESCALA: se sostiene log2(crudo) = ancla - n

## Colocación de las colombianas por títulos predichos

Ruta alternativa a la trilateración genética: los títulos predichos por el modelo `sub` sitúan cada cepa con los sueros fijos en el mapa.

In [6]:
# ===========================================================================
# Colocacion de las secuencias colombianas por sus TITULOS PREDICHOS
#
# Esta es la ruta alternativa a la trilateracion por distancia genetica (que
# la validacion leave-one-out ya dio por negativa). Aqui no interviene ninguna
# distancia genetica: cada cepa se situa donde mejor satisfacen sus titulos
# predichos por el modelo sub de augur, con los SUEROS FIJOS en el mapa.
#
#     d_j = base_j - log2(titulo_predicho_j)      y se minimiza
#     SUM_j ( ||x - suero_j|| - d_j )^2
#
# Por defecto se usan las coordenadas PUBLICADAS (mapa_ag), que son el sistema
# del resto del notebook y en el que estan tus projected_coordinates_*.csv.
# El mapa reajustado da el mismo resultado -mismas posiciones unicas, RMSE
# equivalente, desplazamientos de 0.08-0.38 AU, muy por debajo del estres del
# mapa- asi que la eleccion no cambia ninguna conclusion; se deja conmutable
# para poder comprobarlo.
#
# Dos cautelas aplicadas:
#   1. Se excluyen los sueros con CERO titulaciones medidas: su posicion en el
#      mapa es arbitraria. augur usa uno de ellos (DEN1_VENEZUELA_2000_OBT_1298)
#      y por su culpa el ajuste de DENV1 pasaba de 0.26 a 2.20 AU.
#   2. augur se ajusto por serotipo, asi que cada cepa solo tiene titulos
#      contra sueros de SU serotipo: la posicion queda anclada dentro del
#      cumulo de ese serotipo y no dice nada interserotipico.
# ===========================================================================

from scipy.optimize import minimize

USAR_MAPA = "publicado"        # "publicado" (mapa_ag) | "ajustado" (ACMAP_FIT)

if USAR_MAPA == "publicado":
    _xy = mapa_ag.set_index("strainID")[["coord1", "coord2"]]
    _tipo = mapa_ag.set_index("strainID")["type"]
    SR_xy = _xy[_tipo == "SR"]
    AG_xy = _xy[_tipo == "AG"]
else:
    _fit = pd.read_csv(ACMAP_FIT.format(2))
    SR_xy = _fit[_fit.type == "SR"].set_index("strainID")[["coord1", "coord2"]]
    AG_xy = _fit[_fit.type == "AG"].set_index("strainID")[["coord1", "coord2"]]
print(f"mapa: {USAR_MAPA}  |  {len(AG_xy)} virus, {len(SR_xy)} sueros")

# Cuantas titulaciones MEDIDAS ancla cada suero. Se cuenta sobre la matriz de
# titulos, no sobre el CSV del ajuste, para no depender de que mapa se use.
_n_med = dict(zip(sera, np.bincount(sj[~is_cens], minlength=len(sera))))
sera_nulos = {s for s in sera if _n_med[s] == 0}
print(f"sueros no identificables (0 titulaciones medidas): {sorted(sera_nulos)}")

base_j = dict(zip(sera, basej))        # de la celda de validacion del mapa
pred = pd.read_csv(PRED_TITERS_COL)
n0 = len(pred)
pred = pred[pred.serum_id.isin(SR_xy.index) & ~pred.serum_id.isin(sera_nulos)]
print(f"predicciones usadas: {len(pred)}/{n0}")


def place_point(S, d, n_starts=12, seed=0):
    """Punto que minimiza SUM (||x - s_j|| - d_j)^2, con los sueros fijos."""
    rng = np.random.default_rng(seed)
    f = lambda x: float(((np.linalg.norm(x - S, axis=1) - d) ** 2).sum())
    best, c, r = None, S.mean(0), max(d.max(), 1.0)
    for _ in range(n_starts):
        res = minimize(f, c + rng.normal(scale=r, size=2), method="Nelder-Mead",
                       options=dict(xatol=1e-8, fatol=1e-10, maxiter=4000))
        if best is None or res.fun < best.fun:
            best = res
    return best.x, float(np.sqrt(best.fun / len(d)))


rows = []
for (sero, strain), g in pred.groupby(["serotype", "strain"]):
    S = SR_xy.loc[g.serum_id][["coord1", "coord2"]].to_numpy()
    d = np.array([base_j[s] for s in g.serum_id]) - np.log2(g.pred_titer.to_numpy())
    x, rmse = place_point(S, d)
    rows.append(dict(serotype=sero, strain=strain, coord1=x[0], coord2=x[1],
                     n_sueros=len(d), rmse_ajuste=rmse))
COL = pd.DataFrame(rows)
COL.to_csv(ACMAP_COL_PLACED, index=False)

print("\n" + "=" * 78)
print("RESULTADO: cuantas posiciones distintas produce el modelo?")
print("=" * 78)
print(f"{'':7s} {'cepas':>6s} {'sueros':>7s} {'posiciones':>11s} "
      f"{'dispersion':>11s} {'radio panel':>12s} {'RMSE':>7s}")
resumen = []
for sero, g in COL.groupby("serotype"):
    xy = g[["coord1", "coord2"]].to_numpy()
    uniq = len(np.unique(np.round(xy, 3), axis=0))
    disp = np.sqrt(((xy - xy.mean(0)) ** 2).sum(1)).mean()
    ref = AG_xy.loc[[v for v in AG_xy.index
                     if int(re.search(r"DENV(\d)", v).group(1)) == sero]]
    ref_xy = ref[["coord1", "coord2"]].to_numpy()
    radio = np.sqrt(((ref_xy - ref_xy.mean(0)) ** 2).sum(1)).mean()
    print(f"DENV{sero}  {len(g):6d} {int(g.n_sueros.iloc[0]):7d} {uniq:11d} "
          f"{disp:9.2f} AU {radio:10.2f} AU {g.rmse_ajuste.median():7.3f}")
    resumen.append((sero, len(g), uniq, disp, radio))

print("\n'dispersion' es el radio medio de las posiciones colombianas;")
print("'radio panel' el del panel medido de ese serotipo. La comparacion es el")
print("resultado: las colombianas colapsan en un punto donde el panel real")
print("ocupa un area. El RMSE por debajo del estres del mapa (~0.95 AU) indica")
print("que la colocacion es geometricamente coherente: el problema no es el")
print("ajuste, es que los titulos predichos apenas varian entre cepas.")

# ----------------------------------------------------------------- figura
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
for ax, sero in zip(axes, sorted(COL.serotype.unique())):
    col = SERO_COLORS[sero]
    ref = AG_xy.loc[[v for v in AG_xy.index
                     if int(re.search(r"DENV(\d)", v).group(1)) == sero]]
    ax.scatter(ref.coord1, ref.coord2, s=26, color=col, alpha=0.35,
               linewidths=0, zorder=2)
    srs = SR_xy.loc[[s for s in SR_xy.index
                     if int(re.search(r"DEN(\d)", s).group(1)) == sero
                     and s not in sera_nulos]]
    ax.scatter(srs.coord1, srs.coord2, s=70, facecolors="none", edgecolors=col,
               linewidths=1.5, marker="s", zorder=3)

    g = COL[COL.serotype == sero]
    xy = np.round(g[["coord1", "coord2"]].to_numpy(), 3)
    uniq, cnt = np.unique(xy, axis=0, return_counts=True)
    ax.scatter(uniq[:, 0], uniq[:, 1], s=40 + 170 * cnt / cnt.max(),
               color=INK, marker="X", zorder=5)

    # el numero que resume el resultado, en el mismo sitio en los cuatro paneles
    disp = np.sqrt(((xy - xy.mean(0)) ** 2).sum(1)).mean()
    ref_xy = ref[["coord1", "coord2"]].to_numpy()
    radio = np.sqrt(((ref_xy - ref_xy.mean(0)) ** 2).sum(1)).mean()
    ax.annotate(f"{len(g)} cepas en {len(uniq)} posicion"
                f"{'es' if len(uniq) > 1 else ''}\n"
                f"dispersion    {disp:.2f} AU\n"
                f"panel medido  {radio:.2f} AU",
                xy=(0.03, 0.97), xycoords="axes fraction", va="top",
                fontsize=8, color=INK, family="monospace",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                          edgecolor=GRID, linewidth=0.8))
    ax.set_title(f"DENV{sero}", fontsize=12, color=INK)
    ax.set_aspect("equal")
    ax.set_xlabel("AU", fontsize=9, color=INK_MUTED)
axes[0].set_ylabel("AU", fontsize=9, color=INK_MUTED)

for ax in axes:
    ax.grid(True, color=GRID, linewidth=0.6, alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.tick_params(labelsize=8, colors=INK_MUTED)

handles = [plt.Line2D([], [], marker="o", linestyle="", color=INK_MUTED,
                      alpha=0.45, label="panel medido de Katzelnick"),
           plt.Line2D([], [], marker="s", linestyle="", markerfacecolor="none",
                      markeredgecolor=INK_MUTED, label="sueros (fijos en el mapa)"),
           plt.Line2D([], [], marker="X", linestyle="", color=INK,
                      label="colombianas (tamano = cepas en esa posicion)")]
fig.legend(handles=handles, frameon=False, fontsize=9, ncol=3,
           loc="lower center", bbox_to_anchor=(0.5, -0.06))
fig.tight_layout()

_o = "../figures/acmap_colombia_titulos.png"
fig.savefig(_o, dpi=300, bbox_inches="tight")
print(f"\nfigura -> {_o}")
print(f"coordenadas -> {ACMAP_COL_PLACED}")
plt.show()


mapa: publicado  |  412 virus, 36 sueros


NameError: name 'sj' is not defined